# DE2 — Cleaning pipeline with a cleaning log

**Project:** DS 5020 Mini-Project "Customers Like You" (PCA/SVD recommender on UCI Online Retail II).

The cleaning rules live in `src/clean.py` (`clean(raw_path) -> (clean_df, log_df)`); this notebook only calls that
function, displays the results and writes the artefacts:

- `reports/cleaning_log.csv` — one row per rule: rows before / removed / after + reason (committed)
- `data/processed/clean.parquet` — the cleaned rows (gitignored, regenerated by this notebook)

No train/test split and no customer × product matrix yet — that is the next step.

Rules are applied **in this order**, and each row is counted only under the *first* rule that removes it:

| # | Rule |
|---|------|
| 0 | Normalise types (no removal): `StockCode` → str/strip/upper, `Invoice` → str/strip, `Description` → str, `Customer ID` stays float |
| 1 | Cross-sheet overlap: drop `Year 2010-2011` rows with `InvoiceDate <=` max date of `Year 2009-2010`, then concatenate |
| 2 | Exact duplicate rows (all columns) — keep first |
| 3 | Cancellations (`Invoice` starts with `C`) — not netted against purchases |
| 4 | `Quantity <= 0` |
| 5 | `Price <= 0` |
| 6 | Non-product `StockCode`s (postage, fees, adjustments, vouchers, test rows) |
| 7 | Missing `Customer ID`, then cast to int |

## 0. Setup

Resolve the repo root (works from `notebooks/` or the project root), put it on `sys.path` so `src.clean` imports,
and record the interpreter / library versions used.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.clean import (
    EXPECTED_RAW_ROWS,
    NON_PRODUCT_CODES,
    NON_PRODUCT_PREFIXES,
    clean,
    save_outputs,
    summarise,
)

RAW = ROOT / "data" / "raw" / "online_retail_II.xlsx"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)

print("python  :", sys.executable)
print("pandas  :", pd.__version__)
print("pyarrow :", __import__("pyarrow").__version__)
print("raw     :", RAW.relative_to(ROOT), "| exists:", RAW.exists())
print("rule-6 exclusion list:", sorted(NON_PRODUCT_CODES), "+ prefixes", NON_PRODUCT_PREFIXES)

python  : /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python
pandas  : 3.0.6
pyarrow : 26.0.0
raw     : data/raw/online_retail_II.xlsx | exists: True
rule-6 exclusion list: ['ADJUST', 'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'CRUK', 'D', 'DOT', 'M', 'POST', 'S', 'TEST001'] + prefixes ('GIFT_',)


## 1. Run the pipeline

`clean()` prints a one-line progress summary per rule. During rule 6 it also prints the **full list** of StockCodes
that do not match `^\d{5}[A-Z]*$` (with row count, most common `Description` and the action taken) and, separately,
the non-matching codes that are *not* on the exclusion list and are therefore kept — those are the ones that still
need a human decision. The same tables are captured in `details` and displayed as DataFrames below.

In [2]:
details = {}
clean_df, log_df = clean(RAW, verbose=True, details=details)

[0] normalise types          : 1,067,371 rows (no removal)
[1] cross-sheet overlap      : -  22,523 -> 1,044,848
[2] exact duplicates         : -  11,812 -> 1,033,036


[3] cancellations            : -  19,104 -> 1,013,932
[4] quantity <= 0            : -   3,393 -> 1,010,539
[5] price <= 0               : -   2,626 -> 1,007,913

All StockCodes not matching ^\d{5}[A-Z]*$ at this stage (38 codes):
   StockCode  rows             most_common_description             action
        POST  1851                             POSTAGE drop (non-product)
         DOT  1415                      DOTCOM POSTAGE drop (non-product)
           M   851                              Manual drop (non-product)
          C2   267                            CARRIAGE drop (non-product)
      ADJUST    36 Adjustment by john on 26/01/2010 16 drop (non-product)
BANK CHARGES    34                        Bank Charges drop (non-product)
    DCGS0058    30                    MISO PRETTY  GUM               keep
GIFT_0001_20    26  Dotcomgiftshop Gift Voucher £20.00 drop (non-product)
GIFT_0001_30    24  Dotcomgiftshop Gift Voucher £30.00 drop (non-product)
   DCGSSGIRL    23           

[7] missing customer id      : - 226,761 -> 776,600


## 2. Cleaning log

One row per rule. `rows_removed` is attributed to the first rule that removes a row, so the column sums to
`rows_before[rule 1] - rows_after[rule 7]`. Saved to `reports/cleaning_log.csv`.

In [3]:
display(log_df)

total_removed = int(log_df["rows_removed"].sum())
print(f"rows before rule 1 : {log_df.loc[1, 'rows_before']:,}")
print(f"total rows removed : {total_removed:,}")
print(f"rows after rule 7  : {log_df['rows_after'].iloc[-1]:,}")
assert log_df.loc[1, "rows_before"] - total_removed == log_df["rows_after"].iloc[-1]

,step,rule,rows_before,rows_removed,rows_after,reason
0,0,0 normalise types,1067371,0,1067371,StockCode -> str/strip/upper; Invoice -> str/strip; Description -> str (NaN kept); Customer ID left as float64. No rows removed.
1,1,1 cross-sheet overlap,1067371,22523,1044848,Dropped rows of 'Year 2010-2011' with InvoiceDate <= 2010-12-09 20:01:00 (max of 'Year 2009-2010'); DE1 showed every row in that window is an exact copy of a row in the earlier sheet.
2,2,2 exact duplicates,1044848,11812,1033036,Rows identical in every column (keep first occurrence); same line repeated inside one invoice is an export artefact.
3,3,3 cancellations,1033036,19104,1013932,"Invoice starts with 'C' (8,292 distinct cancellation invoices). Removed without netting against the original purchase."
4,4,4 quantity <= 0,1013932,3393,1010539,"Non-positive quantity outside cancellation invoices (stock adjustments / damages). 3,393 such rows were NOT on a 'C' invoice before rule 3 (DE1 counted 3,457 on the raw data before overlap/duplicate removal)."
5,5,5 price <= 0,1010539,2626,1007913,"Zero price (2,621 rows: free items / missing price) or negative price (5 rows); not a real purchase signal."
6,6,6 non-product stockcodes,1007913,4552,1003361,"Postage, fees, manual/adjustment lines, bank charges, commission, vouchers and test rows: ['ADJUST', 'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'D', 'DOT', 'GIFT_0001_10', 'GIFT_0001_20', 'GIFT_0001_30', 'GIFT_0001_40', 'GIFT_0001_50', 'GIFT_0001_70', 'GIFT_0001_80', 'M', 'POST', 'S', 'TEST001']. DCGS* codes and PADS kept (real products)."
7,7,7 missing customer id,1003361,226761,776600,Rows without a customer cannot be placed in a customer x product matrix. Customer ID then cast to int64.


rows before rule 1 : 1,067,371
total rows removed : 290,771
rows after rule 7  : 776,600


## 3. Non-standard StockCode review (rule 6)

Full table of every StockCode that does not look like a product code at the point rule 6 runs (i.e. after
rules 1–5 have already removed cancellations, non-positive quantities/prices, etc.).

In [4]:
review = details["non_standard_stockcodes"]
print(f"{len(review)} non-matching StockCodes; "
      f"{(review['action'] != 'keep').sum()} dropped by the exclusion list, {(review['action'] == 'keep').sum()} kept")
display(review)

38 non-matching StockCodes; 18 dropped by the exclusion list, 20 kept


,StockCode,rows,most_common_description,action
0,POST,1851,POSTAGE,drop (non-product)
1,DOT,1415,DOTCOM POSTAGE,drop (non-product)
2,M,851,Manual,drop (non-product)
3,C2,267,CARRIAGE,drop (non-product)
4,ADJUST,36,Adjustment by john on 26/01/2010 16,drop (non-product)
5,BANK CHARGES,34,Bank Charges,drop (non-product)
6,DCGS0058,30,MISO PRETTY GUM,keep
7,GIFT_0001_20,26,Dotcomgiftshop Gift Voucher £20.00,drop (non-product)
8,GIFT_0001_30,24,Dotcomgiftshop Gift Voucher £30.00,drop (non-product)
9,DCGSSGIRL,23,GIRLS PARTY BAG,keep


### Non-matching codes NOT on the exclusion list (kept — pending decision)

`DCGS*` codes and `PADS` are deliberately kept (they are real products with real descriptions). Anything else in this
table has not been ruled on yet.

In [5]:
unlisted = details["unlisted_non_standard"]
flag = unlisted[~unlisted["StockCode"].str.startswith("DCGS") & (unlisted["StockCode"] != "PADS")]
display(unlisted)
print("Kept codes that are neither DCGS* nor PADS (need a decision):")
display(flag)

,StockCode,rows,most_common_description,action
0,DCGS0058,30,MISO PRETTY GUM,keep
1,DCGSSGIRL,23,GIRLS PARTY BAG,keep
2,DCGSSBOY,21,BOYS PARTY BAG,keep
3,PADS,17,PADS TO MATCH ALL CUSHIONS,keep
4,DCGS0076,14,SUNJAR LED NIGHT NIGHT LIGHT,keep
5,DCGS0003,13,BOXED GLASS ASHTRAY,keep
6,DCGS0069,5,OOH LA LA DOGS COLLAR,keep
7,DCGS0004,4,HAYNES CAMPER SHOULDER BAG,keep
8,DCGS0072,3,CAT CAMOUFLAGUE COLLAR,keep
9,ADJUST2,3,Adjustment by Peter on Jun 25 2010,keep


Kept codes that are neither DCGS* nor PADS (need a decision):


,StockCode,rows,most_common_description,action
9,ADJUST2,3,Adjustment by Peter on Jun 25 2010,keep
13,SP1002,2,KID'S CHALKBOARD/EASEL,keep
15,TEST002,1,This is a test product.,keep


## 4. Final dataset summary and sanity checks

Row count, distinct customers / products, date range, dtypes, and the required assertion that the row count
entering rule 1 equals the raw total of 1,067,371.

In [6]:
assert log_df.loc[1, "rows_before"] == EXPECTED_RAW_ROWS == 1_067_371, "rows before rule 1 must equal 1,067,371"
print(f"assertion OK: rows before rule 1 == {EXPECTED_RAW_ROWS:,}")
print()

s = summarise(clean_df)
print(f"final rows          : {s['rows']:,}")
print(f"distinct customers  : {s['distinct_customers']:,}")
print(f"distinct products   : {s['distinct_products']:,}")
print(f"distinct invoices   : {s['distinct_invoices']:,}")
print(f"date range          : {s['date_min']}  ->  {s['date_max']}")
print()
print("dtypes after cleaning:")
print(clean_df.dtypes.to_string())
print()
print("remaining missing values per column:")
print(clean_df.isna().sum().to_string())
assert (clean_df["Quantity"] > 0).all() and (clean_df["Price"] > 0).all()
assert not clean_df["Invoice"].str.startswith("C").any()
assert clean_df["Customer ID"].notna().all() and clean_df["Customer ID"].dtype == "int64"

assertion OK: rows before rule 1 == 1,067,371

final rows          : 776,600
distinct customers  : 5,852
distinct products   : 4,623
distinct invoices   : 36,598
date range          : 2009-12-01 07:45:00  ->  2011-12-09 12:50:00

dtypes after cleaning:
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str

remaining missing values per column:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0


In [7]:
clean_df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085,United Kingdom


## 5. Save artefacts

`data/processed/clean.parquet` (gitignored via `data/*`) and `reports/cleaning_log.csv` (committed).
The parquet file is read back to confirm it round-trips.

In [8]:
parquet_path, log_path = save_outputs(clean_df, log_df, ROOT)
print("saved ->", parquet_path.relative_to(ROOT), f"({parquet_path.stat().st_size / 1e6:.1f} MB)")
print("saved ->", log_path.relative_to(ROOT))

check = pd.read_parquet(parquet_path)
assert len(check) == len(clean_df) and list(check.columns) == list(clean_df.columns)
print(f"parquet round-trip OK: {len(check):,} rows, columns {list(check.columns)}")

saved -> data/processed/clean.parquet (5.0 MB)
saved -> reports/cleaning_log.csv


parquet round-trip OK: 776,600 rows, columns ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


## Notes

- Cancellations are removed but the original purchases are **kept**: a bought-then-returned item still signals
  product interest for a recommender.
- Rule 6 only excludes the codes on the explicit list; the "pending decision" table above lists the remaining
  non-standard codes so they can be ruled on in `decision_log.md`.